### Imports


In [ ]:
# !pip install google-genai scipy boto3 -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_fscore_support,
)
import re
import math
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins
import signal
import subprocess
import tempfile
import textwrap

from scipy.stats import binomtest, norm

import multiprocessing
import traceback
import hashlib

from typing import List, Dict, Any, Tuple, Callable, Optional, Union

### Constants


In [ ]:
# Experiment parameters
EXPERIMENT_NUMBER = "expC"
EXP_VERSION = "v1"
GENERATION_TYPE = (
    "during"  # 'during' or 'after' or 'only-gen' or 'without-correctness'
)
DATASET = "humaneval"

In [ ]:
MODEL_NAME = "claude"
MODEL_PATH = "gemini-2.5-flash"

# Project root directory (absolute path to avoid confusion with relative paths)
BASE_DIR = "/home/fahad/Documents/PROJECTS/promptmark"  # or '.'

# Derived paths
if DATASET == "humaneval":
    DATASET_PATH = f"{BASE_DIR}/datasets/humaneval_164.json"
else:
    DATASET_PATH = (
        f"{BASE_DIR}/datasets/sanitized-mbpp-sample-100.json"
    )

# Watermark parameters
Z_THRESHOLD = 2.12  # corresponds to a p-value of approximately 0.017 and confidence level of 98.3%
P_THRESHOLD = norm.sf(Z_THRESHOLD)
# GAMMA = 0.514161

SEED_KEY = "exp2025"
SMALL_SAMPLE_THRESHOLD = 30
N_MIN_TOKENS = 5  # Minimum tokens required for watermark evaluation
ITER_CAP = 5

# ============================================================================
# CONTROL CONSTANTS - Modify these to change experiment behavior
# ============================================================================
COMMENT_ENABLED = True # If True: include comments in generated code; If False: generate code without comments
CHECK_CORRECTNESS = True  # If True: evaluate correctness (tests passed); If False: skip correctness check
APPLY_WATERMARKING = True  # If True: apply watermarking constraints to prompt; If False: generic code generation
ITERATIVE_MODE = False  # If True: generate with feedback loops (iterative); If False: generate once (one-shot)
MINIMAL_TOKEN_CHECK = False  # If True: skip watermark evaluation for outputs with fewer than N_MIN_TOKENS tokens; If False: evaluate all outputs regardless of length
TRIVIAL_APPROACH = True  # If True: use a trivial GREEN-RED SET; If False: use the Adaptive GREEN-RED set
# ============================================================================

In [ ]:
# missing_ids = ['HumanEval/3', 'HumanEval/5', 'HumanEval/8', 'HumanEval/9', 'HumanEval/12', 'HumanEval/13', 'HumanEval/15', 'HumanEval/79', 'HumanEval/127', 'HumanEval/129', 'HumanEval/130', 'HumanEval/132', 'HumanEval/133', 'HumanEval/134', 'HumanEval/135', 'HumanEval/136', 'HumanEval/137', 'HumanEval/138', 'HumanEval/140', 'HumanEval/141', 'HumanEval/143', 'HumanEval/144', 'HumanEval/145', 'HumanEval/146', 'HumanEval/147', 'HumanEval/149', 'HumanEval/150', 'HumanEval/151', 'HumanEval/152', 'HumanEval/153', 'HumanEval/154', 'HumanEval/155', 'HumanEval/156', 'HumanEval/157', 'HumanEval/158', 'HumanEval/159', 'HumanEval/160', 'HumanEval/161', 'HumanEval/162', 'HumanEval/163']

In [ ]:
df = pd.read_json(DATASET_PATH, lines=True)


# df = df[5:5+2]

# sample up to 20 records reproducibly using SEED_KEY
# n = min(2, len(df))
# seed = int.from_bytes(hashlib.sha256(SEED_KEY.encode()).digest()[:4], "big")
# df = df.sample(n=n, random_state=seed).reset_index(drop=True)

print("Columns: ", df.columns.to_list())
print("Size: ", len(df))

In [ ]:
SAMPLE_SIZE = df.shape[0]

RESULTS_CSV = f"{BASE_DIR}/results/raw/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}.csv"
OUTPUT_DIR = f"{BASE_DIR}/output/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}"

### Adaptive Green Tokens


In [ ]:
import hashlib
import random

def trivial_green_set():
    alphabet = list('abcdefghijklmnopqrstuvwxyz')

    # Use EXPERIMENT_NUMBER as seed
    seed_value = int(hashlib.md5(SEED_KEY.encode()).hexdigest(), 16)
    print("SEED VALUE: ", seed_value)
    random.seed(seed_value)

    # Shuffle the alphabet randomly
    random.shuffle(alphabet)

    # Divide into two equal halves
    half1 = set(alphabet[:13])
    half2 = set(alphabet[13:])

    # Use seed_hash to decide green and red halves
    seed_hash = seed_value % 2

    if seed_hash == 0:
        green_letters = half1
        red_letters = half2
    else:
        green_letters = half2
        red_letters = half1

    return green_letters, len(green_letters)

g, l = trivial_green_set()
print("Green letters: ", g, l)

In [ ]:
import hashlib
import json
import random
from typing import Set, List, Tuple, Dict
from collections import defaultdict

# ============================================================================
# NEW KEY GENERATOR - Frequency-based Green Set Selection
# ============================================================================


def get_frequent_candidates(
    humaneval_freq_file: str, mbpp_freq_file: str, top_n: int = 18
) -> List[str]:
    """
    Extract most frequent characters from both datasets combined.

    Args:
        humaneval_freq_file: Path to humaneval letter frequencies JSON
        mbpp_freq_file: Path to mbpp letter frequencies JSON
        top_n: Number of top characters to return (default: 18)

    Returns:
        List[str]: Characters sorted by combined frequency (highest first)
    """
    freq_sum = defaultdict(int)

    for filepath in [humaneval_freq_file, mbpp_freq_file]:
        if os.path.exists(filepath):
            with open(filepath, "r") as f:
                data = json.load(f)
                for char, count in data["letter_freqs"].items():
                    freq_sum[char] += count
        else:
            print(f"⚠️ Warning: Frequency file not found: {filepath}")

    sorted_chars = sorted(freq_sum.items(), key=lambda x: x[1], reverse=True)
    return [char for char, _ in sorted_chars[:top_n]]


def build_green_set(
    secret_key: str, candidate_letters: List[str], g_min: int = 8, g_max: int = 18
) -> Tuple[Set[str], int]:
    """
    Construct green-set from secret key and candidate letters.

    Derives green-set size from SHA256(key), then generates a deterministic
    Fisher-Yates permutation using SHA256(key || "green-set-selection")
    as seed. Returns the first |G| letters from the permutation.

    Args:
        secret_key: Secret key string
        candidate_letters: Candidate letters sorted by frequency (highest first)
        g_min: Minimum green-set size (default: 8)
        g_max: Maximum green-set size (default: 18)

    Returns:
        Tuple[Set[str], int]: Green-set and its size

    Raises:
        ValueError: If parameters are invalid
    """

    if len(candidate_letters) < g_max:
        raise ValueError(
            f"Candidate set size ({len(candidate_letters)}) must be >= g_max ({g_max})"
        )
    if g_min < 1 or g_max > len(candidate_letters) or g_min > g_max:
        raise ValueError(
            f"Invalid bounds: g_min={g_min}, g_max={g_max}, |C|={len(candidate_letters)}"
        )

    # Derive green-set size from first hash
    h1 = hashlib.sha256(secret_key.encode()).digest()
    u1_int = int.from_bytes(h1[0:4], byteorder="big")
    u1 = u1_int / (2**32)
    size_g = int(g_min + u1 * (g_max - g_min + 1))
    size_g = max(g_min, min(g_max, size_g))

    # Derive permutation seed from second hash
    h2_input = secret_key.encode() + b"green-set-selection-v1"
    h2 = hashlib.sha256(h2_input).digest()
    seed = int.from_bytes(h2[0:8], byteorder="big")

    # Generate Fisher-Yates permutation
    shuffled = candidate_letters.copy()
    random.Random(seed).shuffle(shuffled)

    return set(shuffled[:size_g]), size_g


# Module-level variable to track current green set size
CURRENT_GREEN_SET_SIZE = None


def get_red_green_sets(
    secret_key: str = SEED_KEY, base_dir: str = BASE_DIR
) -> Tuple[set, set, int]:
    """
    Get red and green letter sets using frequency-based selection.

    Args:
        secret_key: Secret key for reproducible selection
        base_dir: Base directory for frequency files

    Returns:
        Tuple[set, set, int]: (green_letters, red_letters, green_set_size)
    """
    global CURRENT_GREEN_SET_SIZE

    humaneval_path = "/home/fahad/Documents/PROJECTS/promptmark/results/dataset/humaneval_letter_freqs.json"
    mbpp_path = "/home/fahad/Documents/PROJECTS/promptmark/results/dataset/mbpp_letter_freqs.json"

    candidates = get_frequent_candidates(humaneval_path, mbpp_path, top_n=18)

    if not candidates:
        # Fallback if files not found
        print("⚠️ Using fallback letter frequencies")
        candidates = [
            "i",
            "r",
            "s",
            "t",
            "n",
            "l",
            "m",
            "e",
            "a",
            "c",
            "d",
            "x",
            "f",
            "p",
            "b",
            "j",
            "g",
            "h",
        ]


    if TRIVIAL_APPROACH:
        # Use trivial approach for green/red sets
        green_set, size_g = trivial_green_set()
        red_set = set(candidates) - green_set
    else:
        green_set, size_g = build_green_set(secret_key, candidates, g_min=6, g_max=10)
        red_set = set(candidates) - green_set

    # Track current green set size globally
    CURRENT_GREEN_SET_SIZE = size_g

    # print(f"Green set size: {size_g}")
    # print(f"Green set: {green_set}")
    # print(f"Red set: {red_set}")

    return green_set, red_set, size_g


def calculate_gamma(
    letter_counts: Dict[str, int], total_count: int, green_letters: set
) -> float:
    """
    ✅ CORRECT APPROACH: Calculate gamma from empirical letter frequencies.

    Gamma (γ) represents the expected proportion of identifiers starting with
    green letters, based on the empirical frequency distribution observed in
    the COMBINED datasets (HumanEval + MBPP).

    Under the null hypothesis H₀:
    - An author writes code following natural naming conventions
    - Letter frequencies follow the empirical distribution observed in training data
    - The probability of a random identifier starting with a green letter is γ

    Key insight:
    - Green/red selection uses the top 18 most frequent letters
    - Gamma quantifies how much those selected letters dominate the data
    - If green letters account for 90% of identifiers, γ = 0.90
    - Then observing 6/6 green tokens is NOT unusual (expected ~5.4)

    Formula:
        γ = Σ(frequency[l] / total_count) for l ∈ green_letters

    Args:
        letter_counts: Dictionary mapping letter -> count from COMBINED datasets
        total_count: Total identifiers from COMBINED datasets
        green_letters: Set of green letters (most frequent)

    Returns:
        float: Empirical proportion of green letters (0.0 to 1.0)
    """
    if total_count == 0:
        return 0.0

    gamma = 0.0

    for letter in green_letters:
        if letter in letter_counts:
            p_letter = letter_counts[letter] / total_count
            gamma += p_letter

    return gamma

### Helper Methods


In [ ]:
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins
import keyword

COMMON_STD_METHODS = {
    "self",
    "re",
    "cls",
    "append",
    "join",
    "dummy_function",
    "find",
    "kwargs",
    "args",
    "range",
    "print",
    "len",
    "dict",
    "list",
    "str",
    "int",
    "float",
    "set",
    "tuple",
    "os",
    "np",
    "subprocess",
    "now",
    "today",
    "timedelta",
    "strptime",
    "date",
    "time",
    "datetime",
    "logging",
    "log",
    "info",
    "debug",
    "error",
    "warning",
    "exception",
    "lower",
    "upper",
    "strip",
    "split",
    "replace",
    "endswith",
    "startswith",
    "append",
    "extend",
    "insert",
    "remove",
    "pop",
    "sort",
    "clear",
    "keys",
    "values",
    "items",
    "get",
    "update",
    "copy",
    "format",
    "join",
    "count",
    "index",
}

BUILTIN_NAMES = set(dir(builtins)).union(COMMON_STD_METHODS)


class CodeNavigator(ast.NodeVisitor):
    def __init__(self):
        self.public_classes = set()
        self.non_public_classes = set()
        self.public_funcs = set()
        self.non_public_funcs = set()
        self.public_vars = set()
        self.non_public_vars = set()
        self.docstrings = []

    def visit_FunctionDef(self, node):
        name = node.name

        # classify
        if name.startswith("__") and name.endswith("__"):
            pass
        elif name.startswith("_"):
            self.non_public_funcs.add(name)
        else:
            self.public_funcs.add(name)

        # add arguments
        for arg in node.args.args:
            if arg.arg not in BUILTIN_NAMES:
                self.non_public_vars.add(arg.arg)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "function_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_ClassDef(self, node):
        if node.name.startswith("_"):
            self.non_public_classes.add(node.name)
        else:
            self.public_classes.add(node.name)
        self.non_public_vars.add(node.name)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "class_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id not in BUILTIN_NAMES:
                if target.id.isupper():
                    self.public_vars.add(target.id)
                else:
                    self.non_public_vars.add(target.id)
        self.generic_visit(node)

    def visit_Attribute(self, node):
        # Detect method calls like self.get_possible_moves
        if isinstance(node.value, ast.Name) and node.value.id == "self":
            if node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
                # treat as a public method
                self.public_funcs.add(node.attr)
        elif node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
            # treat other attributes normally
            self.non_public_vars.add(node.attr)
        self.generic_visit(node)

    def visit_Name(self, node):
        if node.id not in BUILTIN_NAMES:
            self.non_public_vars.add(node.id)
        self.generic_visit(node)

    def visit_Module(self, node):
        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "module_docstring",
                    "name": "__main__",
                    "line_number": getattr(node, "lineno", 1),
                    "content": doc,
                }
            )
        self.generic_visit(node)


def get_tokens(code):
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return set()

    visitor = CodeNavigator()
    visitor.visit(tree)

    all_tokens = (
        visitor.public_classes
        # | visitor.public_funcs
        | visitor.non_public_funcs
        | visitor.non_public_vars
        | visitor.non_public_classes
    )

    # 🚫 Remove known stdlib or logging-related names
    cleaned_tokens = {
        t for t in all_tokens if t not in COMMON_STD_METHODS and t not in BUILTIN_NAMES
    }

    return cleaned_tokens


def load_generated_code(file_path):
    if not os.path.exists(file_path):
        return None
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    return content.strip() if content.strip() else None


#! test generated output


def extract_assertions_from_check_function(check_code: str) -> List[str]:
    """Extract all assert statements from a check function."""
    try:
        tree = ast.parse(check_code)
        assertions = []

        for node in ast.walk(tree):
            if isinstance(node, ast.Assert):
                assertion_code = ast.unparse(node)
                assertions.append(assertion_code)

        return assertions
    except:
        return []


def rename_function_to_candidate(code: str) -> str:
    """Rename the first function definition to 'candidate'."""
    try:
        tree = ast.parse(code)

        class FunctionRenamer(ast.NodeTransformer):
            def __init__(self):
                self.renamed = False

            def visit_FunctionDef(self, node):
                if not self.renamed:
                    node.name = "candidate"
                    self.renamed = True
                return node

        renamer = FunctionRenamer()
        tree = renamer.visit(tree)
        ast.fix_missing_locations(tree)
        return ast.unparse(tree)
    except:
        return code


def run_code_with_tests(
    code: str, test_imports: List[str], tests: List[str], return_dict: Dict
) -> Dict:
    passed, failed = 0, 0
    return_dict["error"] = ""
    return_dict["result"] = (passed, failed)
    assertions_list = []

    try:
        code = textwrap.dedent(code)
        code = rename_function_to_candidate(code)
        env = {}

        for imp in test_imports:
            exec(imp, env, env)

        exec(code, env, env)

        for test_idx, test in enumerate(tests, 1):
            try:
                exec(test, env, env)
                passed += 1
                assertions_list.append(
                    {"test_num": test_idx, "assertion": test, "status": "PASSED"}
                )
            except AssertionError as e:
                failed += 1
                error_msg = str(e) if str(e) else "Assertion failed"
                assertions_list.append(
                    {
                        "test_num": test_idx,
                        "assertion": test,
                        "status": "FAILED",
                        "message": error_msg,
                    }
                )
                return_dict["error"] += f"{test}\n{error_msg}\n"
            except Exception as e:
                failed += 1
                assertions_list.append(
                    {
                        "test_num": test_idx,
                        "assertion": test,
                        "status": "ERROR",
                        "message": str(e),
                    }
                )
                return_dict["error"] += f"{test}\n{str(e)}\n"

        return_dict["result"] = (passed, failed)
        return_dict["assertions"] = str(assertions_list)

    except Exception as e:
        tb = traceback.format_exc()
        return_dict["result"] = (0, len(tests))
        return_dict["error"] = tb
        return_dict["assertions"] = str([])

    return return_dict


def safe_exec_with_tests(
    code: str, test_imports: List[str], tests: List[str], timeout: float = 2
) -> Dict[str, Any]:
    manager = multiprocessing.Manager()
    return_dict = manager.dict()
    p = multiprocessing.Process(
        target=run_code_with_tests, args=(code, test_imports, tests, return_dict)
    )

    p.start()
    p.join(timeout)

    if p.is_alive():
        p.terminate()
        return_dict["result"] = (0, len(tests))
        return_dict["error"] = "Timeout: Code execution exceeded time limit"
        return_dict["assertions"] = str([])
        return dict(return_dict)

    result_dict = dict(return_dict)

    if "assertions" in result_dict:
        try:
            result_dict["assertions"] = eval(result_dict["assertions"])
        except:
            result_dict["assertions"] = []

    return result_dict


def test_code(
    code: str, test_imports: List[str], tests: List[str], timeout: float = 2
) -> Dict:
    return safe_exec_with_tests(code, test_imports, tests, timeout)


#! Extract starting letters from comments
def extract_comments_from_source(source_code: str) -> List[Dict[str, Any]]:
    comments = []

    # create a deepcopy
    import copy

    source_code = copy.deepcopy(source_code)

    # Split into lines to process each line individually
    lines = source_code.split("\n")

    for line_number, line in enumerate(lines, 1):
        # Find all # symbols and extract comments after them
        hash_index = line.find("#")
        if hash_index != -1:
            # Extract everything after the # symbol
            comment_content = line[hash_index + 1 :].strip()
            if comment_content:  # Skip empty comments
                # Determine if it's an inline comment or full-line comment
                code_before_hash = line[:hash_index].strip()
                comment_type = (
                    "inline_comment" if code_before_hash else "full_line_comment"
                )

                comments.append(
                    {
                        "line_number": line_number,
                        "content": comment_content,
                        "type": comment_type,
                        "code_context": (
                            code_before_hash[:50] + "..."
                            if len(code_before_hash) > 50
                            else code_before_hash
                        ),
                    }
                )
    # Also extract docstrings using AST visitor
    tree = ast.parse(source_code)
    visitor = CodeNavigator()
    visitor.visit(tree)

    docstrings = visitor.docstrings

    comments.extend(docstrings)

    return comments


def get_comment_starting_letters(source_code: str, gamma) -> tuple:

    try:
        comments = extract_comments_from_source(source_code)

        # print(f"EXTRACTED COMMENTS:\n {[comment['content'] for comment in comments]}\n")

        starting_letters = []
        relevant_words = set()

        for comment in comments:
            # Split comment content by newlines to handle multi-line comments
            comment_lines = comment["content"].split("\n")

            for line in comment_lines:
                line = line.strip()
                if not line:
                    continue

                # Extract the first word from this line
                words = re.findall(r"\b[a-zA-Z]+\b", line)
                if words:
                    first_word = words[0].lower()
                    relevant_words.add(first_word)

                    # Get starting letter of the first word
                    if first_word:
                        first_char = first_word[0].lower()
                        if first_char.isalpha():
                            starting_letters.append(first_char)

        # print(f"Relevant words: {len(relevant_words)}")

        # Use global green_letters and gamma
        green_letters, red_letters, _ = get_red_green_sets(
            secret_key=SEED_KEY, base_dir=BASE_DIR
        )
        green_count = sum(1 for letter in starting_letters if letter in green_letters)
        token_count = len(starting_letters)

        if token_count > 0:
            p_hat = green_count / token_count
            z_score = (p_hat - gamma) / (
                (gamma * (1 - gamma)) ** 0.5 / token_count**0.5
            )
            p_value = norm.sf(z_score)  # one-tailed test
        else:
            z_score = 0.0
            p_value = 1.0

        return starting_letters, relevant_words, green_count, z_score, p_value

    except Exception as e:
        print(f"❌ Error extracting comment letters: {type(e).__name__}: {e}")
        import traceback

        traceback.print_exc()
        return [], set(), 0, 0.0, 1.0


#! fix the method name with the name mentioned in assert statement


def extract_function_names_from_code(code: str):
    """Extract all function names defined in the user code."""
    tree = ast.parse(code)
    return [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]


def extract_function_name_from_tests(test_list):
    """Extract the function name used in assert statements."""
    for test in test_list:
        try:
            tree = ast.parse(test)
            for node in ast.walk(tree):
                # Detect function call inside assert or math.isclose( func(...) )
                if isinstance(node, ast.Call):
                    # Handle nested calls like math.isclose(func(...))
                    for arg in node.args:
                        if isinstance(arg, ast.Call) and isinstance(arg.func, ast.Name):
                            return arg.func.id
                    if isinstance(node.func, ast.Name):
                        return node.func.id
        except SyntaxError:
            continue
    return None


def replace_function_name(code, old_name, new_name):
    """Safely rename the function in the code using AST."""

    class RenameTransformer(ast.NodeTransformer):
        def visit_FunctionDef(self, node):
            if node.name == old_name:
                node.name = new_name
            return node

    tree = ast.parse(code)
    tree = RenameTransformer().visit(tree)
    ast.fix_missing_locations(tree)
    return ast.unparse(tree)


def fix_method_name(user_code, test_list):
    """If needed, rename user's function to match test case function name."""
    code_func_names = extract_function_names_from_code(user_code)
    test_func_name = extract_function_name_from_tests(test_list)

    if not test_func_name:
        print("⚠️ No function found in test cases.")
        return user_code

    # Case 1: If test function name already exists in code, no change needed
    if test_func_name in code_func_names:
        return user_code

    # Case 2: Otherwise, rename the first user function to match test case
    if code_func_names:
        old_name = code_func_names[0]
        updated_code = replace_function_name(user_code, old_name, test_func_name)
        print(f"🔄 Renamed '{old_name}' → '{test_func_name}'")
        return updated_code

    print("⚠️ No function found in user code.")
    return user_code


# ============================================================================
# TEST EXECUTION FUNCTIONS FOR BOTH HUMANEVAL AND MBPP
# ============================================================================

def run_code_with_tests(code, test_imports, tests, return_dict):
    """Execute code with test assertions and track results."""
    try:
        # Shared environment for both code and tests
        env = {}

        # Run any imports from test_imports
        for imp in test_imports:
            exec(imp, env, env)

        # Execute user code
        exec(code, env, env)

        passed, failed = 0, 0
        return_dict["error"] = ""  # Initialize error as empty string

        # Run all test assertions
        for t in tests:
            try:
                exec(t, env, env)
                passed += 1
            except AssertionError:
                failed += 1
                return_dict["error"] += f"Assertion Error in: {t}\n"
            except Exception as e:
                failed += 1
                return_dict["error"] += f"Exception Error in: {t} → {e}\n"

        return_dict["result"] = (passed, failed)

    except Exception as e:
        tb = traceback.format_exc()
        return_dict["error"] = f"Runtime Error in user code:\n{tb}"

    return return_dict


def safe_exec_with_tests(code, test_imports, tests, timeout=2):
    """
    Safely execute code with tests using multiprocessing for timeout handling.
    Works for both HumanEval and MBPP datasets.
    """
    manager = multiprocessing.Manager()
    return_dict = manager.dict()
    p = multiprocessing.Process(
        target=run_code_with_tests, args=(code, test_imports, tests, return_dict)
    )

    p.start()
    p.join(timeout)

    if p.is_alive():
        p.terminate()
        print("⏰ Timeout: possible infinite loop or hanging code.")
        return_dict["result"] = (0, len(tests))
        return_dict["error"] = "Timeout: possible infinite loop or hanging code"
        return return_dict

    if "error" in return_dict:
        return return_dict

    return return_dict


def extract_assertions_from_check_function(test_code_str, entry_point_name=None):
    """
    Extract assertion statements from a HumanEval check() function.
    Replace 'candidate' calls with the actual function name if entry_point_name is provided.
    
    HumanEval format: def check(candidate)...assert candidate(...)...; 
    Returns: List of assertion strings that can be directly executed.
    """
    assertions = []
    
    try:
        tree = ast.parse(test_code_str)
        
        # Find the check() function if it exists
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef) and node.name == "check":
                # Extract assertions from the function body
                for stmt in node.body:
                    if isinstance(stmt, ast.Assert):
                        # Convert AST assertion back to source code string
                        assert_code = ast.unparse(stmt)
                        
                        # Replace 'candidate' with actual function name if provided
                        if entry_point_name:
                            assert_code = assert_code.replace('candidate(', f'{entry_point_name}(')
                        
                        assertions.append(assert_code)
                break
        
        return assertions
    except SyntaxError:
        return []


def _get_tests_from_record(record):
    """Return (test_imports, tests_list) from a record safely for both MBPP and HumanEval."""
    
    # For MBPP: has test_imports and test_list
    if "test_list" in record and record.get("test_list"):
        test_imports = record.get("test_imports", []) or record.get("imports", []) or []
        tests_list = record.get("test_list", []) or record.get("tests", []) or []
    # For HumanEval: has test field with check() function
    elif "test" in record and record.get("test"):
        test_code_str = record.get("test", "")
        test_imports = [
            line.strip()
            for line in test_code_str.split("\n")
            if line.strip().startswith(("import ", "from "))
        ]
        # Extract entry_point name for HumanEval (used to replace 'candidate' in assertions)
        entry_point = record.get("entry_point", None)
        tests_list = extract_assertions_from_check_function(test_code_str, entry_point_name=entry_point)
    else:
        test_imports = []
        tests_list = []
    
    # Normalize to list of strings
    if isinstance(test_imports, str):
        test_imports = [test_imports]
    if isinstance(tests_list, str):
        tests_list = [tests_list]
    
    return test_imports, tests_list



def test_code(code, test_imports, tests, timeout=2):
    """
    Test generated code against test cases.
    Wrapper around safe_exec_with_tests with proper error handling.
    """
    return safe_exec_with_tests(code, test_imports, tests, timeout=timeout)

In [ ]:
import json
from datetime import datetime

FAILURE_LOG_PATH = f"{MODEL_NAME}_{EXPERIMENT_NUMBER}_{EXP_VERSION}_failures_{datetime.now().strftime('%Y%m%d_%H_%M_%S')}.json"


def log_failure(record_id, failure_type, error_details):
    """Append failure details into a JSON file."""
    log_entry = {
        "task_id": record_id,
        "failure_type": failure_type,
        "error_details": error_details,
        "timestamp": datetime.now().isoformat(),
    }

    try:
        # Load existing log file
        if os.path.exists(FAILURE_LOG_PATH):
            with open(FAILURE_LOG_PATH, "r") as f:
                data = json.load(f)
        else:
            data = []

        data.append(log_entry)

        with open(FAILURE_LOG_PATH, "w") as f:
            json.dump(data, f, indent=4)

    except Exception as e:
        print(f"[Warning] Failed to log error: {e}")

In [ ]:
import math
from scipy.stats import binomtest, norm


def calculate_z_score(token_count, green_count, gamma):
    """Calculate z-score for green token count (standard normal approximation)."""
    if token_count == 0 or gamma <= 0 or gamma >= 1:
        return float("nan")
    return (green_count - gamma * token_count) / math.sqrt(
        gamma * (1 - gamma) * token_count
    )


def calculate_p_value_exact(green_count, token_count, gamma):
    """
    Calculate exact binomial p-value (one-sided test: H1: p > gamma).
    Use this for ALL samples; it's exact and always valid.
    """
    if token_count == 0:
        return float("nan")
    return binomtest(green_count, token_count, gamma, alternative="greater").pvalue


def calculate_p_value_normal(z_score):
    """
    Calculate approximate p-value using standard normal CDF (one-sided).
    Used only for reference/large samples where normal approx is valid.
    """
    if math.isnan(z_score):
        return 1.0
    return norm.sf(z_score)


from scipy.stats import binomtest, norm, binom, chi2


def get_unified_detection_score(token_count, green_count, gamma):
    """
    ✅ SIMPLIFIED: Use only exact binomial p-value (no z-test).

    Returns p-exact and unified score.
    - p_exact = binom.sf(green_count - 1, token_count, gamma)  (exact binomial test)
    - score = -log10(p_exact)  (for ROC analysis and ranking)

    This approach is:
    - Mathematically exact for all sample sizes (not just approximations)
    - Simpler (no z-test, no sample size selection logic)
    - Consistent across all tasks
    """
    p_exact = binom.sf(green_count - 1, token_count, gamma)

    # Score for ROC: higher = more evidence of watermark
    score = -np.log10(np.clip(p_exact, 1e-300, 1.0))

    return {
        "p_exact": p_exact,
        "score": score,
        "token_count": token_count,
    }

In [ ]:
def detect_watermark(original_code, generated_code, green_letters, red_letters, gamma):
    """
    ✅ FIXED: UNIFIED WATERMARK DETECTION WITH PROPER DEDUPLICATION

    - Extracts identifier tokens and comment tokens SEPARATELY
    - Merges with proper deduplication to prevent double-counting
    - Single unified p-exact score for decision and ROC ranking
    - Returns separate metrics for transparency but calculates ONE p-value

    Key insight on deduplication:
    - If 'total' is both an identifier AND in a comment, it counts as 2 tokens
    - But we track: id_token_count + comment_token_count - overlap_count
    - Green count follows the same deduplication logic

    ✅ FIX: Dedent code before parsing to handle indented canonical solutions
    """
    import ast

    def get_starting_letters(code):
        # ✅ FIX: Dedent the code before parsing to handle indented code snippets
        code = textwrap.dedent(code)

        try:
            tree = ast.parse(code)
        except SyntaxError:
            return {
                "id_starting_letters": [],
                "id_unique_tokens": set(),
                "id_green_count": 0,
                "id_token_count": 0,
                "comment_starting_letters": [],
                "comment_unique_tokens": set(),
                "comment_green_count": 0,
                "comment_token_count": 0,
                "total_starting_letters": [],
                "total_green_count": 0,
                "total_token_count": 0,
                "overlap_count": 0,
                "p_exact": 1.0,
                "score": 0.0,
                "z_score": 0.0,
            }

        visitor = CodeNavigator()
        visitor.visit(tree)

        # ===== PHASE 1: IDENTIFIER TOKENS =====
        non_public_tokens = (
            visitor.non_public_classes
            | visitor.non_public_funcs
            | visitor.non_public_vars
        )

        relevant_tokens = [
            token for token in non_public_tokens if token not in {"self", "cls"}
        ]

        def get_starting_letter(word):
            if not word:
                return None
            if word[0] == "_":
                if len(word) > 1 and word[1].isalpha():
                    return word[1].lower()
                else:
                    return None
            elif word[0].isalpha():
                return word[0].lower()
            else:
                return None

        id_starting_letters = [
            letter
            for word in relevant_tokens
            if (letter := get_starting_letter(word)) is not None
        ]

        id_green_count = sum(
            1 for letter in id_starting_letters if letter in green_letters
        )
        id_token_count = len(id_starting_letters)
        id_unique_tokens = set(relevant_tokens)

        # ===== PHASE 2: COMMENT TOKENS =====
        comment_data = {}
        comment_starting_letters = []
        comment_unique_tokens = set()
        comment_green_count = 0

        if COMMENT_ENABLED:
            cmnt_starting_letters, cmn_relevant_words, cmnt_green_count, _, _ = (
                get_comment_starting_letters(code, gamma)
            )
            comment_starting_letters = cmnt_starting_letters
            comment_unique_tokens = set(cmn_relevant_words)
            comment_green_count = cmnt_green_count

        comment_token_count = len(comment_starting_letters)

        # ===== PHASE 3: MERGE WITH DEDUPLICATION =====
        # Count overlap: tokens that appear in both identifiers and comments
        overlap_tokens = id_unique_tokens & comment_unique_tokens

        # Count how many starting letters correspond to overlap
        # For overlap tokens, we DON'T double-count them
        # Instead: they contribute to total_token_count and total_green_count once each

        # Simple approach: just concatenate the starting letters
        # This naturally accounts for the actual occurrences in each phase
        total_starting_letters = id_starting_letters + comment_starting_letters
        total_green_count = id_green_count + comment_green_count
        total_token_count = id_token_count + comment_token_count

        # Number of distinct token names that appear in both phases
        overlap_count = len(overlap_tokens)

        # ===== CALCULATE p_exact, p_unified, z_score, AND SCORE =====
        if total_token_count == 0:
            p_exact = 1.0
            z_score = 0.0
            score = 0.0
        else:
            detection_stats = get_unified_detection_score(
                total_token_count, total_green_count, gamma
            )
            p_exact = detection_stats["p_exact"]
            score = detection_stats["score"]
            z_score = calculate_z_score(total_token_count, total_green_count, gamma)

        return {
            # Identifier metrics
            "id_starting_letters": id_starting_letters,
            "id_unique_tokens": id_unique_tokens,
            "id_green_count": id_green_count,
            "id_token_count": id_token_count,
            # Comment metrics
            "comment_starting_letters": comment_starting_letters,
            "comment_unique_tokens": comment_unique_tokens,
            "comment_green_count": comment_green_count,
            "comment_token_count": comment_token_count,
            # Combined metrics
            "total_starting_letters": total_starting_letters,
            "total_green_count": total_green_count,
            "total_token_count": total_token_count,
            "overlap_count": overlap_count,
            # Detection stats
            "p_exact": p_exact,
            "z_score": z_score,
            "score": score,
        }

    orig_stats = get_starting_letters(original_code)

    gen_stats = get_starting_letters(generated_code)

    # ✅ UNIFIED DETECTION: Use p_exact for both decision and all returned stats
    def is_watermarked_unified(p_exact, threshold=P_THRESHOLD):
        """Simple threshold-based decision using p_exact."""
        if math.isnan(p_exact):
            return False
        return p_exact < threshold

    return {
        # Original code stats - matching exact original CSV column format
        "original_token_count": orig_stats.get("total_token_count", 0),
        "original_green_count": orig_stats.get("total_green_count", 0),
        "original_z_score": orig_stats.get("z_score", 0.0),
        "original_p_exact": orig_stats.get("p_exact", 1.0),
        "original_p_unified": orig_stats.get("p_exact", 1.0),  # Use same as p_exact
        "original_score": orig_stats.get("score", 0.0),
        "original_method_used": "binomial",
        "original_preferred_ratio": orig_stats.get("total_green_count", 0)
        / max(1, orig_stats.get("total_token_count", 1)),
        "original_is_watermarked": is_watermarked_unified(
            orig_stats.get("p_exact", 1.0)
        ),
        "original_unique_starts": "".join(
            sorted(set(orig_stats.get("id_starting_letters", [])))
        ),
        # Generated code stats - matching exact original CSV column format
        "generated_token_count": gen_stats.get("total_token_count", 0),
        "generated_green_count": gen_stats.get("total_green_count", 0),
        "generated_z_score": gen_stats.get("z_score", 0.0),
        "generated_p_exact": gen_stats.get("p_exact", 1.0),
        "generated_p_unified": gen_stats.get("p_exact", 1.0),  # Use same as p_exact
        "generated_score": gen_stats.get("score", 0.0),
        "generated_method_used": "binomial",
        "generated_preferred_ratio": gen_stats.get("total_green_count", 0)
        / max(1, gen_stats.get("total_token_count", 1)),
        "generated_is_watermarked": is_watermarked_unified(
            gen_stats.get("p_exact", 1.0)
        ),
        "generated_unique_starts": "".join(
            sorted(set(gen_stats.get("id_starting_letters", [])))
        ),
    }

### PROMPT TEMPLATE

```tex
Wang, C. Y., DaghighFarsoodeh, A., & Pham, H. V. (2024).
Selection of prompt engineering techniques for code generation through predicting code complexity.
arXiv preprint arXiv:2409.16416.```
```


In [ ]:
SYSTEM_PROMPT = '''
### Green Letter List: {green_words}
### Red Letter List: {red_words}

### Command:
Generate code following the given instructions:
    1. Green Letter Enriched Identifier: When generating identifiers (local variables, function parameters, private helper functions, internal class attributes, and temporary variables) prefer those starting with letters from the 'Green Letter List'. Use them naturally and consistently.
    2. Correct & Relevant: Generate correct code following the problem statement.
    2. Avoiding Instruction: Do not add docstrings. Add brief comments only to clarify complex logic, but do not over-comment. Reduce the use of Red List letters.
    3. Important: Write the method named according to the given test case.
    4. Warning: Do not mention or explain the renaming rules in your output.

### Example Identifier names:
    - Preferred (Green List): answer, count, index, value, sum, key, item, name, word, var, input, output, obj, attr, param, arg, var1, var2, temp_var, helper
    - To Avoid (Red List): result, temp, data, list, flag, ptr, elem, hash, dict, res, tmp, dat, lst, flg, p, el, h, d
'''

In [ ]:
# Prompt templates
PROBLEM_TEMPLATE = (
    "You are a helpful code assistant who can teach a junior developer how to code. Your language of choice is Python. Only generate the Python code for the following task enclosed in ```python```\n\n"
    "##Prompt:\n{prompt}\n\n"
    "##Test Cases:\n{tests}\n\n"
)

### Generate Response

In [ ]:
import boto3


def generate_response(
    prompt: str,
    max_tokens: int = 2048,
    temperature: float = 0.1,
    track_tokens: bool = False,
):
    """Generate response from Claude via AWS Bedrock with optional token tracking."""
    client = boto3.client(
        "bedrock-runtime",
        region_name=os.getenv('AWS_REGION', 'us-east-1'),
        aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
        aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
    )

    model_id = os.getenv("DEFAULT_MODEL", "us.anthropic.claude-sonnet-4-20250514-v1:0")

    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
    }

    response = client.invoke_model(
        modelId=model_id, body=json.dumps(request_body), contentType="application/json"
    )

    response_body = json.loads(response["body"].read())
    text = ""
    if "content" in response_body and len(response_body["content"]) > 0:
        text = response_body["content"][0]["text"]

    if track_tokens:
        usage = response_body.get("usage", {})
        return {
            "text": text,
            "input_tokens": usage.get("input_tokens", 0),
            "output_tokens": usage.get("output_tokens", 0),
            "total_tokens": usage.get("input_tokens", 0)
            + usage.get("output_tokens", 0),
        }
    else:
        return text

In [ ]:
def generate_code(record, feedback_prompt=""):
    global green_letters, red_letters

    task_id = record["task_id"]
    problem_query = record["prompt"]
    testcases = "\n".join(record['test_list']) if DATASET == 'mbpp' else "\n".join(extract_assertions_from_check_function(record["test"]))
    full_prompt = PROBLEM_TEMPLATE.format(prompt=problem_query, tests=testcases)

    # Choose system prompt based on APPLY_WATERMARKING setting
    if APPLY_WATERMARKING:
        green_letters, red_letters, _ = get_red_green_sets(
            secret_key=SEED_KEY, base_dir=BASE_DIR
        )
        system_instruction = SYSTEM_PROMPT.format(
            green_words=green_letters, red_words=red_letters
        )
    else:
        # Use generic prompt without watermarking constraints
        system_instruction = ""

    if len(feedback_prompt) > 0:
        full_prompt = full_prompt + "\n\n" + feedback_prompt

    full_prompt_with_system = f"{system_instruction}\n\n{full_prompt}"

    print(f"FULL PROMPT: {full_prompt_with_system}\n")

    result = generate_response(
        full_prompt_with_system, max_tokens=2048, track_tokens=True
    )

    full_text = result["text"].strip()
    input_tokens = result["input_tokens"]
    output_tokens = result["output_tokens"]

    code_blocks = re.findall(r"```python(.*?)```", full_text, re.DOTALL)
    actual_code_blocks = [block.strip() for block in code_blocks if block.strip()]
    code_text = actual_code_blocks[-1] if actual_code_blocks else ""
    explanation_text = re.sub(
        r"```python.*?```", "", full_text, flags=re.DOTALL
    ).strip()

    return {
        "code": code_text,
        "explanation": explanation_text,
        "full_response": full_text,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
    }

### Execution


In [ ]:
def _load_frequency_data(green_letters, base_dir):

    # humaneval freqs
    he_freq_path = os.path.join(base_dir, "results/dataset/humaneval_letter_freqs.json")
    mbpp_freq_path = os.path.join(base_dir, "results/dataset/mbpp_letter_freqs.json")

    letter_freqs = defaultdict(int)
    total_identifiers = 0

    if os.path.exists(he_freq_path):
        with open(he_freq_path, "r") as f:
            df_he = json.load(f)
            df_he = df_he["letter_freqs"]
            # print('Values: ', df_he)

            for letter in green_letters:
                letter_freqs[letter] += df_he.get(letter, 0)
            total_identifiers += sum(df_he.values())

    if os.path.exists(mbpp_freq_path):
        with open(mbpp_freq_path, "r") as f:
            df_mbpp = json.load(f)
            df_mbpp = df_mbpp["letter_freqs"]

            for letter in green_letters:
                letter_freqs[letter] += df_mbpp.get(letter, 0)
            total_identifiers += sum(df_mbpp.values())

    return letter_freqs, total_identifiers

In [ ]:
def evaluate_candidate(record, generated_code):
    """
    Evaluate generated code for:
    1. Correctness (does it pass tests?) - if CHECK_CORRECTNESS is True
    2. Watermark fidelity (p-values, z-scores) - if CHECK_CORRECTNESS is True OR APPLY_WATERMARKING is True

    Handles both HumanEval and MBPP datasets:
    - HumanEval: Uses "test" field with check() function
    - MBPP: Uses "test_list" and "test_imports" fields
    
    Returns dict with keys matching the original CSV format:
    - task_id, tests_passed, tests_failed, total_tests, pass_rate, all_passed
    - original_* and generated_* columns (z_score, p_exact, p_unified, score, method_used, preferred_ratio, token_count, green_count, is_watermarked, unique_starts)
    """
    global CURRENT_GREEN_SET_SIZE

    task_id = record["task_id"]
    
    # Detect dataset type and extract tests accordingly
    is_mbpp = "test_list" in record
    is_humaneval = "test" in record
    
    # Evaluate correctness only if CHECK_CORRECTNESS is True
    if CHECK_CORRECTNESS:
        # Get tests from record (handles both MBPP and HumanEval)
        test_imports, tests = _get_tests_from_record(record)
        
        # For MBPP: fix function name in generated code to match test expectations
        if is_mbpp and tests:
            generated_code = fix_method_name(generated_code, tests)
        
        test_result = test_code(generated_code, test_imports, tests, timeout=2)
        passed, failed = test_result.get("result", (0, 0))
    else:
        # Skip correctness check
        passed, failed = 0, 0
        test_result = {"error": "Correctness check disabled"}

    is_correct = (failed == 0) if CHECK_CORRECTNESS else None

    total_tests = passed + failed if CHECK_CORRECTNESS else 0
    pass_rate = (
        (passed / total_tests * 100.0)
        if (CHECK_CORRECTNESS and total_tests > 0)
        else 0.0
    )
    all_passed = (failed == 0) if CHECK_CORRECTNESS else None

    eval_res = {
        "task_id": task_id,
        "tests_passed": passed,
        "tests_failed": failed,
        "total_tests": total_tests,
        "pass_rate": pass_rate,
        "all_passed": all_passed if CHECK_CORRECTNESS else False,
        "correctness": is_correct if CHECK_CORRECTNESS else None,
        "error_message": test_result.get("error", ""),
    }

    # Calculate watermark metrics if EITHER CHECK_CORRECTNESS or APPLY_WATERMARKING is True
    # This ensures we always have p-values and z-scores when checking correctness
    should_calculate_watermark = CHECK_CORRECTNESS or APPLY_WATERMARKING

    if should_calculate_watermark:
        # Get original code (handles both HumanEval and MBPP)
        original_code = record.get("canonical_solution", "") or record.get("code", "")
        green_letters, red_letters, size_g = get_red_green_sets(
            secret_key=SEED_KEY, base_dir=BASE_DIR
        )
        CURRENT_GREEN_SET_SIZE = size_g

        letter_freqs, total_identifiers = _load_frequency_data(green_letters, BASE_DIR)
        # GAMMA = calculate_gamma(letter_freqs, total_identifiers, green_letters)
        GAMMA = 0.39
        print(f"⚠️ Calculated GAMMA: {GAMMA:.4f} based on frequency data")

        # Estimate token count early to decide on watermark evaluation
        try:
            tree = ast.parse(generated_code)
            visitor = CodeNavigator()
            visitor.visit(tree)
            estimated_tokens = len(
                visitor.non_public_vars
                | visitor.non_public_funcs
                | visitor.non_public_classes
            )
        except:
            estimated_tokens = 0

        # Check if we have minimum tokens for watermark detection
        # Note: Only skip if APPLY_WATERMARKING is True; if just CHECK_CORRECTNESS, always calculate
        if APPLY_WATERMARKING and estimated_tokens < N_MIN_TOKENS and MINIMAL_TOKEN_CHECK:
            print(
                f"⏭️  [Insufficient tokens: {estimated_tokens} < {N_MIN_TOKENS}] Skipping watermark analysis"
            )
            # Return base result with NaN/None for watermark fields
            eval_res.update(
                {
                    "original_z_score": float("nan"),
                    "original_p_exact": float("nan"),
                    "original_p_unified": float("nan"),
                    "original_score": float("nan"),
                    "original_method_used": "binomial",
                    "original_preferred_ratio": float("nan"),
                    "original_token_count": 0,
                    "original_green_count": 0,
                    "original_is_watermarked": False,
                    "meets_z": False,
                    "original_unique_starts": "",
                    "generated_z_score": float("nan"),
                    "generated_p_exact": float("nan"),
                    "generated_p_unified": float("nan"),
                    "generated_score": float("nan"),
                    "generated_method_used": "binomial",
                    "generated_preferred_ratio": float("nan"),
                    "generated_token_count": estimated_tokens,
                    "generated_green_count": 0,
                    "generated_is_watermarked": False,
                    "generated_unique_starts": "",
                }
            )
            return eval_res

        # Full watermark detection (calculates p-values, z-scores for both original and generated code)
        detection_result = detect_watermark(
            original_code, generated_code, green_letters, red_letters, GAMMA
        )

        # Add all watermark stats to results
        eval_res.update(detection_result)

        # Only set meets_z if APPLY_WATERMARKING is True (this affects the stopping condition)
        # But we always calculate the metrics
        eval_res["meets_z"] = (
            bool(detection_result.get("generated_is_watermarked", False))
            if APPLY_WATERMARKING
            else False
        )
    else:
        # Neither correctness checking nor watermarking - skip all watermark detection
        eval_res.update(
            {
                "original_z_score": float("nan"),
                "original_p_exact": float("nan"),
                "original_p_unified": float("nan"),
                "original_score": float("nan"),
                "original_method_used": "none",
                "original_preferred_ratio": float("nan"),
                "original_token_count": 0,
                "original_green_count": 0,
                "original_is_watermarked": False,
                "meets_z": False,
                "original_unique_starts": "",
                "generated_z_score": float("nan"),
                "generated_p_exact": float("nan"),
                "generated_p_unified": float("nan"),
                "generated_score": float("nan"),
                "generated_method_used": "none",
                "generated_preferred_ratio": float("nan"),
                "generated_token_count": 0,
                "generated_green_count": 0,
                "generated_is_watermarked": False,
                "generated_unique_starts": "",
            }
        )

    return eval_res

In [ ]:
def run_phase1(
    record, max_iterations=ITER_CAP, verbose=False, iterative_mode=ITERATIVE_MODE
):
    """
    Generate code following the configured behavior:

    If iterative_mode=True (ITERATIVE):
    - CHECK_CORRECTNESS=True, APPLY_WATERMARKING=True: iterate until correct AND watermarked
    - CHECK_CORRECTNESS=True, APPLY_WATERMARKING=False: iterate until correct (no watermarking)
    - CHECK_CORRECTNESS=False, APPLY_WATERMARKING=True: iterate until watermarked (no correctness check)
    - CHECK_CORRECTNESS=False, APPLY_WATERMARKING=False: generate once (simple generation)

    If iterative_mode=False (ONE-SHOT):
    - Generate code once without feedback loops
    - No iterative refinement, regardless of CHECK_CORRECTNESS or APPLY_WATERMARKING settings

    Args:
        record: Task record with prompt and test information
        max_iterations: Maximum number of iterations to attempt (ignored if iterative_mode=False)
        verbose: Whether to print verbose output
        iterative_mode: If True, use iterative generation with feedback; if False, generate once

    Returns: (selected_result_dict, token_tracking_dict)
    """

    best = None
    selected = None
    token_tracking = {"iterations": []}
    feedback_prompt = ""  # Accumulate feedback across iterations
    green_letters, red_letters, _ = (
        get_red_green_sets(secret_key=SEED_KEY, base_dir=BASE_DIR)
        if APPLY_WATERMARKING
        else (set(), set(), 0)
    )

    # In one-shot mode, only generate once regardless of conditions
    iterations_to_run = 1 if not iterative_mode else max_iterations

    for it in range(iterations_to_run):
        print(f"\n{'='*60}\n[Iteration {it+1}/{iterations_to_run}]\n{'='*60}")

        # --- Generate code ---
        gen = generate_code(record, feedback_prompt=feedback_prompt)
        code = gen["code"]
        reasoning_text = gen.get("explanation", "")

        iter_tokens = {
            "iteration": it,
            "input_tokens": gen.get("input_tokens", 0),
            "output_tokens": gen.get("output_tokens", 0),
            "total_tokens": gen.get("total_tokens", 0),
        }
        token_tracking["iterations"].append(iter_tokens)

        # print(">> RESPONSE CODE:\n", code)

        eval_res = evaluate_candidate(record, code)
        eval_res.update(
            {
                "iteration": it,
                "reasoning_text": reasoning_text,
                "full_llm_response": gen["full_response"],
                "input_tokens": iter_tokens["input_tokens"],
                "output_tokens": iter_tokens["output_tokens"],
            }
        )

        # Determine stopping condition based on control constants
        if iterative_mode:
            # Iterative mode: check stopping conditions
            if CHECK_CORRECTNESS and APPLY_WATERMARKING:
                # Both correctness and watermarking required
                eval_res["stopping_condition_met"] = (
                    eval_res["correctness"] and eval_res["meets_z"]
                )
            elif CHECK_CORRECTNESS and not APPLY_WATERMARKING:
                # Only correctness required (no watermarking)
                eval_res["stopping_condition_met"] = eval_res["correctness"]
            elif not CHECK_CORRECTNESS and APPLY_WATERMARKING:
                # Only watermarking required (no correctness check)
                eval_res["stopping_condition_met"] = eval_res["meets_z"]
            else:
                # No checking required (CHECK_CORRECTNESS=False, APPLY_WATERMARKING=False)
                # Just generate once
                eval_res["stopping_condition_met"] = True
        else:
            # One-shot mode: always stop after first generation
            eval_res["stopping_condition_met"] = True

        eval_res["code"] = code

        # ✅ DEBUG: Print evaluation metrics
        print(f"\n📊 EVALUATION METRICS:")
        if CHECK_CORRECTNESS:
            print(f"   Correctness: {eval_res['correctness']}")
            print(
                f"   Tests Passed: {eval_res['tests_passed']}/{eval_res['tests_passed'] + eval_res['tests_failed']}"
            )
        if APPLY_WATERMARKING:
            print(f"   Generated Token Count: {eval_res['generated_token_count']}")
            print(f"   Generated Green Count: {eval_res['generated_green_count']}")
            print(f"   P-Value (p_exact): {eval_res['generated_p_exact']:.10f}")
            print(f"   Score (-log10(p)): {eval_res['generated_score']:.4f}")
            print(f"   Meets Watermark (p < {P_THRESHOLD:.6f}): {eval_res['meets_z']}")
        print(f"   Stopping Condition Met: {eval_res['stopping_condition_met']}\n")

        # --- Save reasoning if failure ---
        if not eval_res["stopping_condition_met"]:
            failure_entry = {
                "task_id": record["task_id"],
                "iteration": it,
                "timestamp": datetime.now().isoformat(),
                "correctness": (
                    bool(eval_res["correctness"]) if CHECK_CORRECTNESS else None
                ),
                "meets_z": bool(eval_res["meets_z"]) if APPLY_WATERMARKING else None,
                "error_message": eval_res.get("error_message", ""),
                "llm_reasoning": reasoning_text,
                "input_tokens": iter_tokens["input_tokens"],
                "output_tokens": iter_tokens["output_tokens"],
            }
            # FAILURE_LOG.append(failure_entry)

        # --- Prepare feedback if not stopping ---
        feedback_prompt = ""

        # Early stopping
        if eval_res["stopping_condition_met"]:
            selected = eval_res
            token_tracking["total_input_tokens"] = sum(
                t["input_tokens"] for t in token_tracking["iterations"]
            )
            token_tracking["total_output_tokens"] = sum(
                t["output_tokens"] for t in token_tracking["iterations"]
            )
            token_tracking["total_tokens"] = (
                token_tracking["total_input_tokens"]
                + token_tracking["total_output_tokens"]
            )
            if verbose:
                msg = f"[{record['task_id']}] ✅ Stop: "
                if CHECK_CORRECTNESS:
                    msg += f"Correct ({eval_res['tests_passed']}/{eval_res['total_tests']})"
                if APPLY_WATERMARKING:
                    if CHECK_CORRECTNESS:
                        msg += " & "
                    p_val = eval_res.get("generated_p_exact", float("nan"))
                    msg += f"p_exact={p_val:.6f} < {P_THRESHOLD}"
                print(msg)
            break

        # Only prepare feedback in iterative mode
        if iterative_mode:
            # Prepare feedback based on failures
            if CHECK_CORRECTNESS and not eval_res["correctness"]:
                error_log = eval_res.get("error_message", "Unknown error")
                feedback_prompt = f"❌ The previous code execution failed.\nAnalyze the error step by step:\n1. Review the error log: \n{error_log}\n2. Consider the previous reasoning: {reasoning_text}\n\nNow write the explanation why it failed. \nThen generate the corrected version of the code that fixes the issues. \nEnsure the code is executable following the problem requirements."
                print(f"\n🔄 FEEDBACK REASON: Code correctness failed")
            elif APPLY_WATERMARKING and not eval_res["meets_z"]:
                # Watermark fidelity failed - code is correct but doesn't have watermark
                p_exact = eval_res.get("generated_p_exact", 1.0)
                token_count = eval_res.get("generated_token_count", 0)
                green_count = eval_res.get("generated_green_count", 0)
                score = eval_res.get("generated_score", float("nan"))

                feedback_prompt = f"Your provided code is correct BUT FAILED to create WATERMARK (p-value={p_exact:.6f}, P_THRESHOLD={P_THRESHOLD:.6f}).\nGreen Token Count={green_count}/{token_count}. Only REFACTOR the code to use more green-letter identifiers ({', '.join(list(green_letters))}) in identifiers and comments.\n"
                print(f"\n🔄 FEEDBACK REASON: Watermark fidelity failed")

            if feedback_prompt:
                print(f"\n💬 FEEDBACK PROMPT:\n{feedback_prompt}\n")

        # Update best result (using unified p-value score for ranking: higher score = stronger watermark)
        if APPLY_WATERMARKING:
            current_score = eval_res.get("generated_score", -1e9)
            if math.isnan(current_score):
                current_score = -1e9
            best_score = best.get("generated_score", -1e9) if best else -1e9
            if math.isnan(best_score):
                best_score = -1e9

            if best is None or (
                (
                    eval_res.get("correctness", False)
                    and not best.get("correctness", False)
                )
                if CHECK_CORRECTNESS
                else True or (current_score > best_score)
            ):
                best = eval_res
        else:
            # If not applying watermarking, track best by correctness
            if CHECK_CORRECTNESS:
                if best is None or (
                    eval_res.get("correctness", False)
                    and not best.get("correctness", False)
                ):
                    best = eval_res
            else:
                best = eval_res

    if selected is None:
        selected = best
        token_tracking["total_input_tokens"] = sum(
            t["input_tokens"] for t in token_tracking["iterations"]
        )
        token_tracking["total_output_tokens"] = sum(
            t["output_tokens"] for t in token_tracking["iterations"]
        )
        token_tracking["total_tokens"] = (
            token_tracking["total_input_tokens"] + token_tracking["total_output_tokens"]
        )

    return (selected or {}, token_tracking)

In [ ]:
import time


def process_dataset(df, output_dir):
    Path(output_dir).parent.mkdir(exist_ok=True)
    results = []
    all_token_tracking = []
    total_exp_input_tokens = 0
    total_exp_output_tokens = 0

    for idx, record in df.iterrows():
        task_id = record.get("task_id") or record.get("id")
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        try:
            sel, token_info = run_phase1(
                record, max_iterations=ITER_CAP, verbose=True, iterative_mode=ITERATIVE_MODE
            )
            code = sel.get("code", "") if sel else ""
            iteration_used = sel.get("iteration") if sel and "iteration" in sel else None

            # Save code
            output_file = out_dir / f"{task_id}.py"

                
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(code or "")
            
            # Track tokens
            total_exp_input_tokens += token_info.get("total_input_tokens", 0)
            total_exp_output_tokens += token_info.get("total_output_tokens", 0)
            
            # Store results
            result_row = {
                "task_id": task_id,
                "iteration_used": iteration_used,
                **sel
            }
            results.append(result_row)
            all_token_tracking.append({
                "task_id": task_id,
                **token_info
            })
            
        except Exception as e:
            print(f"❌ Error processing {task_id}: {e}")
            import traceback
            traceback.print_exc()
    
    # Save results to CSV
    results_df = pd.DataFrame(results)
    results_df.to_csv(RESULTS_CSV, index=False)
    print(f"\n✅ Results saved to {RESULTS_CSV}")
    print(f"📊 Total tokens used: {total_exp_input_tokens + total_exp_output_tokens}")
    
    return results_df

### RUN


In [ ]:
results = process_dataset(df, OUTPUT_DIR)

### TEST COMBINED FUNCTIONALITY WITH BOTH DATASETS